# SignSense AI — MLP Training Notebook

**Model:** MLP landmark classifier (ASL A–Z + space/del/nothing = 29 classes)  
**Input:** 63-dim normalized MediaPipe landmark vector  
**Architecture:** Input(63) → Dense(512) → BN → ReLU → Dropout → Dense(256) → ... → Softmax(29)  
**Target accuracy:** > 95%  
**Runtime:** ~15 min on Colab T4 GPU

---
### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your Kaggle API key (`kaggle.json`) when prompted, OR mount Google Drive with the dataset already downloaded
3. Run all cells top to bottom

In [2]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create model output directory on Drive
import os
DRIVE_MODELS_DIR = '/content/drive/MyDrive/SignSense/models'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
print(f'Drive mounted. Models will be saved to: {DRIVE_MODELS_DIR}')

Mounted at /content/drive
Drive mounted. Models will be saved to: /content/drive/MyDrive/SignSense/models


In [3]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# Colab already has TensorFlow + NumPy. Install the extras we need.
!pip install -q mediapipe==0.10.14 scikit-learn tqdm albumentations
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 39.1 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 23.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
Dependencies installed.


In [ ]:
# ── Cell 3: Upload backend code to Colab ────────────────────────────────────
# Step 1: Run this PowerShell script LOCALLY to create the zip:
#   .\notebooks\create_colab_zip.ps1
#
# Step 2: Upload backend_colab.zip using the button below
#   (Files panel on the left → Upload, OR run the cell to get a file picker)

from google.colab import files
import os, sys, zipfile

BACKEND_PATH = '/content/backend'

if not os.path.exists(BACKEND_PATH):
    print('Upload backend_colab.zip when the file picker appears...')
    uploaded = files.upload()  # opens file picker
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content')
    print(f'Extracted {zip_name} to /content/')
else:
    print('backend/ already exists, skipping upload.')

# Add backend to Python path
sys.path.insert(0, BACKEND_PATH)

# Verify
from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'Import OK — {NUM_CLASSES} classes: {ASL_CLASSES[:5]}...')

Upload backend_colab.zip when the file picker appears...


In [ ]:
# ── Cell 4: Download & preprocess ASL dataset ─────────────────────────────────
#
# Prerequisites:
#   - Cell 3 must have run (backend/ is at /content/backend)
#   - Upload kaggle.json via Files panel (left sidebar) BEFORE running this cell
#     Get it from: https://www.kaggle.com/settings → API → Create New Token
#
# What this cell does:
#   1. Checks if preprocessed data already exists (idempotent)
#   2. Validates kaggle.json is present before attempting download
#   3. Downloads the Kaggle ASL Alphabet dataset (~1GB)
#   4. Auto-detects the actual extracted directory structure
#   5. Copies raw images into the correct directory structure
#   6. Runs preprocess.py to extract landmarks + augment
#   7. Verifies the output .npy files

import os, sys, shutil
import numpy as np
from pathlib import Path

BACKEND_PATH   = '/content/backend'
RAW_ASL_DIR    = f'{BACKEND_PATH}/data/raw/ASL'
PROCESSED_DIR  = f'{BACKEND_PATH}/data/processed/ASL'
PROCESSED_NPY  = f'{PROCESSED_DIR}/landmarks_all.npy'
LABELS_NPY     = f'{PROCESSED_DIR}/labels_all.npy'
KAGGLE_JSON    = '/content/kaggle.json'
KAGGLE_DEST    = '/root/.kaggle/kaggle.json'
DOWNLOAD_DIR   = '/content/asl_data'

# ASL classes we expect (29 total)
ASL_CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ') + ['space', 'del', 'nothing']

# ── Guard: backend must be uploaded first ────────────────────────────────────
if not os.path.exists(BACKEND_PATH):
    raise RuntimeError(
        'backend/ not found at /content/backend.\n'
        'Run Cell 3 first to upload backend_colab.zip.'
    )

# ── Check if preprocessing already done ──────────────────────────────────────
if os.path.exists(PROCESSED_NPY) and os.path.exists(LABELS_NPY):
    print('✅ Preprocessed data already exists — skipping download and preprocessing.')
else:
    # ── Guard: kaggle.json must be uploaded ──────────────────────────────────
    if not os.path.exists(KAGGLE_JSON):
        raise FileNotFoundError(
            'kaggle.json not found at /content/kaggle.json.\n'
            'Steps to fix:\n'
            '  1. Go to https://www.kaggle.com/settings → API → Create New Token\n'
            '  2. This downloads kaggle.json to your computer\n'
            '  3. In Colab: Files panel (left sidebar) → Upload → select kaggle.json\n'
            '  4. Re-run this cell'
        )

    # ── Install kaggle CLI ────────────────────────────────────────────────────
    print('Installing kaggle CLI...')
    !pip install -q kaggle

    # ── Configure kaggle credentials ─────────────────────────────────────────
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp {KAGGLE_JSON} {KAGGLE_DEST}
    !chmod 600 {KAGGLE_DEST}
    print('Kaggle credentials configured.')

    # ── Download dataset ──────────────────────────────────────────────────────
    print('Downloading Kaggle ASL Alphabet dataset (~1GB)...')
    !kaggle datasets download -d grassknoted/asl-alphabet -p {DOWNLOAD_DIR} --unzip

    # ── Auto-detect actual extracted structure ────────────────────────────────
    # The zip may extract to different layouts depending on the dataset version:
    #   Layout A: /content/asl_data/asl_alphabet_train/A/, B/, ...
    #   Layout B: /content/asl_data/A/, B/, ...
    #   Layout C: /content/asl_data/asl_alphabet_train/asl_alphabet_train/A/, ...
    # We search for the directory that contains the most ASL class subdirs.

    print('\nDetecting dataset structure...')
    print(f'Contents of {DOWNLOAD_DIR}:')
    for item in sorted(Path(DOWNLOAD_DIR).rglob('*')):
        if item.is_dir():
            n_imgs = len(list(item.glob('*.jpg')))
            if n_imgs > 0 or item.parent == Path(DOWNLOAD_DIR):
                print(f'  {item.relative_to(DOWNLOAD_DIR)}/  ({n_imgs} .jpg files)')

    # Find the directory that contains class subdirs with .jpg images
    TRAIN_DIR = None
    best_count = 0

    for candidate in Path(DOWNLOAD_DIR).rglob('*'):
        if not candidate.is_dir():
            continue
        subdirs = [d for d in candidate.iterdir() if d.is_dir()]
        # Count subdirs that contain .jpg files (class dirs)
        class_like = [d for d in subdirs if len(list(d.glob('*.jpg'))) > 0]
        if len(class_like) > best_count:
            best_count = len(class_like)
            TRAIN_DIR = str(candidate)

    if TRAIN_DIR is None or best_count == 0:
        raise FileNotFoundError(
            f'Could not find class directories with .jpg images under {DOWNLOAD_DIR}.\n'
            'The dataset structure may have changed. Check the Files panel.'
        )

    print(f'\n✅ Detected training directory: {TRAIN_DIR}')
    print(f'   Found {best_count} class directories with images.')

    if best_count < 29:
        print(f'   WARNING: Expected 29 classes, found {best_count}. Continuing anyway.')

    # ── Copy raw images into project structure ────────────────────────────────
    print(f'\nCopying raw images to {RAW_ASL_DIR}...')
    os.makedirs(RAW_ASL_DIR, exist_ok=True)

    class_dirs = [d for d in Path(TRAIN_DIR).iterdir() if d.is_dir()]
    copied_count = 0
    for class_dir in class_dirs:
        dest = Path(RAW_ASL_DIR) / class_dir.name
        if not dest.exists():
            shutil.copytree(str(class_dir), str(dest))
            copied_count += 1
        else:
            copied_count += 1  # already there

    # Verify copy
    total_images = sum(
        len(list(d.glob('*.jpg')))
        for d in Path(RAW_ASL_DIR).iterdir() if d.is_dir()
    )
    print(f'✅ Copied {copied_count} class dirs, {total_images:,} images to {RAW_ASL_DIR}')

    if total_images == 0:
        raise RuntimeError(
            f'No .jpg images found in {RAW_ASL_DIR} after copy.\n'
            f'Source was: {TRAIN_DIR}\n'
            'Check the Files panel to inspect the directory.'
        )

    # ── Run preprocessing pipeline ────────────────────────────────────────────
    print('\nRunning preprocessing pipeline (MediaPipe landmark extraction + augmentation)...')
    print('This takes ~10-20 min depending on dataset size.')
    !python {BACKEND_PATH}/src/preprocess.py --all --augment --aug_factor 3

    # ── Verify preprocessing output ───────────────────────────────────────────
    if not os.path.exists(PROCESSED_NPY):
        raise RuntimeError(
            f'Preprocessing did not produce {PROCESSED_NPY}.\n'
            'Check the preprocess.py output above for errors.'
        )

# ── Load and verify final arrays ──────────────────────────────────────────────
X = np.load(PROCESSED_NPY)
y = np.load(LABELS_NPY)

print(f'\n✅ Data ready:')
print(f'   X shape: {X.shape}  (samples × 63 landmarks)')
print(f'   y shape: {y.shape}  (class indices 0–28)')
print(f'   Classes: {len(set(y.tolist()))} unique')
print(f'   Samples per class (avg): {len(X) // max(len(set(y.tolist())), 1)}')

# Expose for downstream cells
PROCESSED_NPY_PATH = PROCESSED_NPY
LABELS_NPY_PATH    = LABELS_NPY

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU memory growth enabled.')
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')

In [ ]:
# ── Cell 6: Train MLP ─────────────────────────────────────────────────────────
from pathlib import Path
from configs.training_config import MLPConfig
from src.train import train_mlp

cfg = MLPConfig()
cfg.save_dir = Path(DRIVE_MODELS_DIR)
cfg.log_dir  = Path('/content/logs/mlp')
cfg.mixed_precision = True  # Enable on T4 GPU for ~30% speedup

print('Config:')
print(f'  hidden_dims:    {cfg.hidden_dims}')
print(f'  dropout_rate:   {cfg.dropout_rate}')
print(f'  epochs:         {cfg.epochs}')
print(f'  batch_size:     {cfg.batch_size}')
print(f'  learning_rate:  {cfg.learning_rate}')
print(f'  mixed_precision:{cfg.mixed_precision}')
print()

model = train_mlp(cfg)
print('\nTraining complete!')

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
from src.evaluate import evaluate

# Temporarily point MODELS_DIR to Drive so evaluate() finds the model
import configs.training_config as tc
tc.MODELS_DIR = Path(DRIVE_MODELS_DIR)

results = evaluate('asl_mlp', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/logs/mlp

In [ ]:
# ── Cell 9: Verify saved model ────────────────────────────────────────────────
import os
saved_files = os.listdir(DRIVE_MODELS_DIR)
print('Files saved to Drive:')
for f in saved_files:
    size_mb = os.path.getsize(os.path.join(DRIVE_MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

# Quick sanity check: load and run one prediction
import numpy as np
import tensorflow as tf
loaded = tf.keras.models.load_model(os.path.join(DRIVE_MODELS_DIR, 'asl_mlp.keras'))
dummy = np.zeros((1, 63), dtype=np.float32)
pred = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f} (should be ~1.0)')